# LinkedIn CS Job Market Analysis
**Dataset:** [arshkon/linkedin-job-postings](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings) (2023–2024, 124k+ postings)

**Sections:**
0. Data Loading & Cleaning
1. CS Job Filtering & Breakdown
2. Job Listing Trends Over Time
3. Skills Analysis
4. Salary & Compensation
5. Work Type & Remote Trends
6. Experience Level & Geography
7. Company & Demand Stats

## 0. Data Loading & Cleaning

In [ ]:
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = "../data"

# ── Load main files ──────────────────────────────────────────────────────────
jobs = pd.read_csv(f"{DATA_DIR}/job_postings.csv", low_memory=False)
companies = pd.read_csv(f"{DATA_DIR}/company_details/companies.csv", low_memory=False)

print(f"Jobs: {jobs.shape}")
print(f"Companies: {companies.shape}")
jobs.head(3)

In [ ]:
# ── Parse timestamps ─────────────────────────────────────────────────────────
jobs["listed_dt"] = pd.to_datetime(jobs["listed_time"], unit="ms", utc=True).dt.tz_localize(None)
jobs["expiry_dt"] = pd.to_datetime(jobs["expiry"], unit="ms", utc=True, errors="coerce").dt.tz_localize(None)
jobs["month"] = jobs["listed_dt"].dt.to_period("M").astype(str)
jobs["week"] = jobs["listed_dt"].dt.to_period("W").astype(str)

# ── Drop rows missing critical fields ────────────────────────────────────────
jobs = jobs.dropna(subset=["title", "listed_time"])
jobs["title_lower"] = jobs["title"].str.lower()
jobs["skills_desc"] = jobs["skills_desc"].fillna("").str.lower()

# ── Extract US state from location ───────────────────────────────────────────
us_states = [
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA",
    "KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
    "NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT",
    "VA","WA","WV","WI","WY","DC"
]
state_pattern = r',\s*(' + '|'.join(us_states) + r')(?:\s|,|$)'
jobs["state"] = jobs["location"].str.extract(state_pattern, expand=False)

# ── Merge company info ────────────────────────────────────────────────────────
jobs = jobs.merge(companies[["company_id", "name", "company_size"]], on="company_id", how="left")

print(f"Date range: {jobs['listed_dt'].min().date()} → {jobs['listed_dt'].max().date()}")
print(f"Clean job count: {len(jobs):,}")

## 1. CS Job Filtering & Subcategory Breakdown

In [ ]:
# ── CS subcategory keyword mapping ────────────────────────────────────────────
CS_CATEGORIES = {
    "SWE": [
        "software engineer", "software developer", "backend", "back-end",
        "frontend", "front-end", "full stack", "fullstack", "full-stack",
        "web developer", "mobile developer", "ios developer", "android developer",
        "application developer", "programmer"
    ],
    "Data": [
        "data engineer", "data analyst", "data scientist", "analytics engineer",
        "business analyst", "business intelligence", "etl", "data warehouse",
        "data architect"
    ],
    "ML / AI": [
        "machine learning", "deep learning", "nlp", "computer vision",
        "artificial intelligence", " ai ", "ai/ml", "ml engineer",
        "mlops", "llm", "generative ai", "research scientist"
    ],
    "Cloud / Infra": [
        "devops", "sre", "site reliability", "cloud engineer", "platform engineer",
        "infrastructure engineer", "systems engineer", "systems administrator",
        "network engineer", "database administrator", "dba"
    ],
    "Security": [
        "security engineer", "cybersecurity", "information security",
        "appsec", "application security", "penetration", "soc analyst",
        "security analyst", "threat"
    ],
    "Other CS": [
        "technical program manager", "technical product manager",
        "solutions architect", "solutions engineer", "qa engineer",
        "quality assurance", "embedded engineer", "firmware",
        "computer scientist", "it manager"
    ],
}

def classify_job(title: str) -> str:
    for category, keywords in CS_CATEGORIES.items():
        for kw in keywords:
            if kw in title:
                return category
    return "Non-CS"

jobs["category"] = jobs["title_lower"].apply(classify_job)
cs = jobs[jobs["category"] != "Non-CS"].copy()

print(f"CS jobs: {len(cs):,} / {len(jobs):,} ({len(cs)/len(jobs):.1%})")
jobs["category"].value_counts()

In [ ]:
cat_counts = (
    jobs["category"].value_counts()
      .reset_index()
      .rename(columns={"count": "count"})
)

fig = px.bar(
    cat_counts,
    x="category", y="count",
    color="category",
    title="Job Postings by CS Subcategory",
    labels={"category": "Category", "count": "Postings"},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Share of CS subcategories (pie)
cs_counts = (
    cs["category"].value_counts()
      .reset_index()
      .rename(columns={"count": "count"})
)

fig = px.pie(
    cs_counts,
    names="category", values="count",
    title="CS Job Postings — Subcategory Share",
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.show()

## 2. Job Listing Trends Over Time

In [ ]:
# Monthly postings per CS subcategory
monthly = (
    cs.groupby(["month", "category"])
      .size()
      .reset_index(name="count")
      .sort_values("month")
)

fig = px.line(
    monthly,
    x="month", y="count",
    color="category",
    markers=True,
    title="Monthly CS Job Listings by Subcategory",
    labels={"month": "Month", "count": "Job Postings", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# 4-week rolling average for total CS listings
weekly_total = (
    cs.groupby("week")
      .size()
      .reset_index(name="count")
      .sort_values("week")
)
weekly_total["rolling_avg"] = weekly_total["count"].rolling(4, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=weekly_total["week"], y=weekly_total["count"],
    name="Weekly Count", marker_color="lightsteelblue", opacity=0.6
))
fig.add_trace(go.Scatter(
    x=weekly_total["week"], y=weekly_total["rolling_avg"],
    name="4-Week Rolling Avg", line=dict(color="royalblue", width=2.5)
))
fig.update_layout(
    title="Weekly CS Job Listings with 4-Week Rolling Average",
    xaxis_title="Week", yaxis_title="Job Postings",
    xaxis_tickangle=45
)
fig.show()

In [ ]:
# Stacked area: share of each subcategory per month
monthly_pct = monthly.copy()
total_per_month = monthly_pct.groupby("month")["count"].transform("sum")
monthly_pct["pct"] = monthly_pct["count"] / total_per_month * 100

fig = px.area(
    monthly_pct,
    x="month", y="pct",
    color="category",
    title="Monthly Share of CS Subcategories (%)",
    labels={"month": "Month", "pct": "Share (%)", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_xaxes(tickangle=45)
fig.show()

## 3. Skills Analysis

In [ ]:
# ── Master skills keyword list ────────────────────────────────────────────────
SKILLS = [
    # Languages
    "python", "java", "javascript", "typescript", "c++", "c#", "go", "golang",
    "rust", "scala", "kotlin", "swift", "r ", "ruby", "php", "bash", "shell",
    # Web
    "react", "angular", "vue", "node.js", "django", "flask", "fastapi", "spring",
    "html", "css",
    # Data & ML
    "sql", "spark", "hadoop", "kafka", "airflow", "dbt", "pandas", "numpy",
    "scikit-learn", "tensorflow", "pytorch", "keras", "hugging face", "langchain",
    "tableau", "power bi", "looker",
    # Cloud & Infra
    "aws", "azure", "gcp", "google cloud", "kubernetes", "docker", "terraform",
    "ansible", "jenkins", "ci/cd", "linux", "git",
    # Databases
    "postgresql", "mysql", "mongodb", "redis", "elasticsearch", "dynamodb",
    "snowflake", "redshift", "bigquery",
    # Soft / Domain
    "agile", "scrum", "rest api", "graphql", "microservices",
]

def extract_skills(text: str) -> list:
    return [skill for skill in SKILLS if skill in text]

cs["skills_list"] = cs["skills_desc"].apply(extract_skills)

# Flatten to (job_id, skill) pairs
skill_rows = cs[["job_id", "category", "month", "skills_list"]].explode("skills_list").dropna(subset=["skills_list"])
skill_rows = skill_rows.rename(columns={"skills_list": "skill"})

print(f"Skill mentions: {len(skill_rows):,}")
print(f"Jobs with ≥1 skill: {skill_rows['job_id'].nunique():,}")

In [ ]:
# Top 25 skills — horizontal bar
top_skills = (
    skill_rows["skill"]
      .value_counts()
      .head(25)
      .reset_index()
      .rename(columns={"count": "count"})
)

fig = px.bar(
    top_skills[::-1],          # reverse so highest is at top
    x="count", y="skill",
    orientation="h",
    title="Top 25 Skills Across CS Job Postings",
    labels={"skill": "Skill", "count": "Postings Mentioning Skill"},
    color="count",
    color_continuous_scale="Blues"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Skills heatmap: top 15 skills × CS subcategory (normalized by category posting count)
top15 = skill_rows["skill"].value_counts().head(15).index.tolist()
cat_counts_map = cs["category"].value_counts().to_dict()

heat_data = (
    skill_rows[skill_rows["skill"].isin(top15)]
      .groupby(["category", "skill"])
      .size()
      .reset_index(name="count")
)
heat_data["rate"] = heat_data.apply(
    lambda r: r["count"] / cat_counts_map.get(r["category"], 1) * 100, axis=1
)

heat_pivot = heat_data.pivot(index="skill", columns="category", values="rate").fillna(0)

fig = go.Figure(go.Heatmap(
    z=heat_pivot.values,
    x=heat_pivot.columns.tolist(),
    y=heat_pivot.index.tolist(),
    colorscale="Blues",
    text=heat_pivot.values.round(1),
    texttemplate="%{text}%",
    colorbar=dict(title="% of Postings")
))
fig.update_layout(
    title="Skill Demand Rate by CS Subcategory (% of postings in category)",
    xaxis_title="CS Subcategory",
    yaxis_title="Skill"
)
fig.show()

In [ ]:
# Skill trend over time — top 10 skills mention rate per month
top10 = skill_rows["skill"].value_counts().head(10).index.tolist()
monthly_jobs = cs.groupby("month").size().rename("total")

skill_trend = (
    skill_rows[skill_rows["skill"].isin(top10)]
      .groupby(["month", "skill"])
      .size()
      .reset_index(name="count")
      .merge(monthly_jobs, on="month")
)
skill_trend["rate"] = skill_trend["count"] / skill_trend["total"] * 100

fig = px.line(
    skill_trend.sort_values("month"),
    x="month", y="rate",
    color="skill",
    markers=True,
    title="Top 10 Skills — Monthly Mention Rate (% of CS Postings)",
    labels={"month": "Month", "rate": "Mention Rate (%)", "skill": "Skill"},
    color_discrete_sequence=px.colors.qualitative.Vivid
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Skills co-occurrence matrix (top 15 skills)
top15_skills = skill_rows["skill"].value_counts().head(15).index.tolist()
pivot = (
    skill_rows[skill_rows["skill"].isin(top15_skills)]
      .assign(present=1)
      .pivot_table(index="job_id", columns="skill", values="present", fill_value=0)
)
cooc = pivot.T.dot(pivot)
# Zero diagonal
import numpy as np
np.fill_diagonal(cooc.values, 0)

fig = go.Figure(go.Heatmap(
    z=cooc.values,
    x=cooc.columns.tolist(),
    y=cooc.index.tolist(),
    colorscale="Purples",
    colorbar=dict(title="Co-occurrences")
))
fig.update_layout(
    title="Skill Co-occurrence Matrix (Top 15 Skills)",
    xaxis_tickangle=45
)
fig.show()

## 4. Salary & Compensation

In [ ]:
# Normalize salary to annual based on pay_period
PAY_MULTIPLIERS = {
    "HOURLY": 2080,
    "WEEKLY": 52,
    "BIWEEKLY": 26,
    "MONTHLY": 12,
    "YEARLY": 1,
    "ANNUAL": 1,
}

sal = cs.copy()
sal["pay_upper"] = sal["pay_period"].str.upper()
sal["multiplier"] = sal["pay_upper"].map(PAY_MULTIPLIERS).fillna(1)
sal["annual_med"] = sal["med_salary"] * sal["multiplier"]
sal["annual_min"] = sal["min_salary"] * sal["multiplier"]
sal["annual_max"] = sal["max_salary"] * sal["multiplier"]

# Keep plausible annual salaries
sal = sal[(sal["annual_med"] >= 20_000) & (sal["annual_med"] <= 600_000)].dropna(subset=["annual_med"])
print(f"Postings with salary data: {len(sal):,}")

In [ ]:
# Box plot: annual median salary by subcategory
fig = px.box(
    sal,
    x="category", y="annual_med",
    color="category",
    title="Annual Median Salary Distribution by CS Subcategory",
    labels={"category": "Subcategory", "annual_med": "Annual Median Salary ($)"},
    color_discrete_sequence=px.colors.qualitative.Bold,
    points="outliers"
)
fig.update_layout(showlegend=False)
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.show()

In [ ]:
# Salary range: min vs max per category
sal_summary = sal.groupby("category").agg(
    min_sal=("annual_min", "median"),
    med_sal=("annual_med", "median"),
    max_sal=("annual_max", "median")
).reset_index()

fig = go.Figure()
for _, row in sal_summary.iterrows():
    fig.add_trace(go.Bar(
        name=row["category"],
        x=[row["category"]],
        y=[row["med_sal"]],
        error_y=dict(
            type="data",
            symmetric=False,
            array=[row["max_sal"] - row["med_sal"]],
            arrayminus=[row["med_sal"] - row["min_sal"]]
        )
    ))
fig.update_layout(
    title="Median Salary with Min–Max Range by CS Subcategory",
    yaxis_title="Annual Salary ($)",
    xaxis_title="Category",
    showlegend=False,
    yaxis=dict(tickprefix="$", tickformat=",.0f")
)
fig.show()

In [ ]:
# Salary by remote vs on-site
sal_remote = sal.copy()
sal_remote["work_mode"] = sal_remote["remote_allowed"].map({1.0: "Remote", 0.0: "On-site"}).fillna("Unspecified")

fig = px.violin(
    sal_remote[sal_remote["work_mode"] != "Unspecified"],
    x="work_mode", y="annual_med",
    color="work_mode",
    box=True,
    title="Salary Distribution: Remote vs On-site",
    labels={"work_mode": "Work Mode", "annual_med": "Annual Median Salary ($)"},
    color_discrete_map={"Remote": "royalblue", "On-site": "tomato"}
)
fig.update_layout(showlegend=False)
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.show()

## 5. Work Type & Remote Trends

In [ ]:
# Work type breakdown
wt_counts = (
    cs["formatted_work_type"]
      .value_counts()
      .reset_index()
      .rename(columns={"count": "count"})
)

fig = px.pie(
    wt_counts,
    names="formatted_work_type", values="count",
    title="CS Jobs by Work Type",
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.show()

In [ ]:
# Remote vs on-site monthly trend
cs_remote = cs.copy()
cs_remote["work_mode"] = cs_remote["remote_allowed"].map({1.0: "Remote", 0.0: "On-site"}).fillna("Unspecified")

remote_monthly = (
    cs_remote.groupby(["month", "work_mode"])
      .size()
      .reset_index(name="count")
      .sort_values("month")
)

fig = px.line(
    remote_monthly,
    x="month", y="count",
    color="work_mode",
    markers=True,
    title="Remote vs On-site CS Job Listings Over Time",
    labels={"month": "Month", "count": "Postings", "work_mode": "Work Mode"},
    color_discrete_map={"Remote": "royalblue", "On-site": "tomato", "Unspecified": "gray"}
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Remote rate per CS subcategory
remote_rate = (
    cs.groupby("category")["remote_allowed"]
      .mean()
      .mul(100)
      .reset_index()
      .rename(columns={"remote_allowed": "remote_pct"})
      .sort_values("remote_pct", ascending=True)
)

fig = px.bar(
    remote_rate,
    x="remote_pct", y="category",
    orientation="h",
    title="Remote Work Rate by CS Subcategory",
    labels={"remote_pct": "% Postings Allowing Remote", "category": "Subcategory"},
    color="remote_pct",
    color_continuous_scale="Blues"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 6. Experience Level & Geography

In [ ]:
# Experience level distribution per subcategory
exp = (
    cs[cs["formatted_experience_level"].notna()]
      .groupby(["category", "formatted_experience_level"])
      .size()
      .reset_index(name="count")
)

fig = px.bar(
    exp,
    x="category", y="count",
    color="formatted_experience_level",
    barmode="group",
    title="Experience Level Distribution by CS Subcategory",
    labels={"category": "Subcategory", "count": "Postings", "formatted_experience_level": "Level"},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

In [ ]:
# Experience level as % of subcategory (stacked 100%)
exp_pct = exp.copy()
total_per_cat = exp_pct.groupby("category")["count"].transform("sum")
exp_pct["pct"] = exp_pct["count"] / total_per_cat * 100

fig = px.bar(
    exp_pct,
    x="category", y="pct",
    color="formatted_experience_level",
    barmode="stack",
    title="Experience Level Share by CS Subcategory (%)",
    labels={"category": "Subcategory", "pct": "Share (%)", "formatted_experience_level": "Level"},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

In [ ]:
# US choropleth — CS job postings by state
state_counts = (
    cs[cs["state"].notna()]
      .groupby("state")
      .size()
      .reset_index(name="count")
)

fig = px.choropleth(
    state_counts,
    locations="state",
    locationmode="USA-states",
    color="count",
    scope="usa",
    title="CS Job Postings by US State",
    color_continuous_scale="Blues",
    labels={"count": "Postings"}
)
fig.show()

In [ ]:
# Top 15 cities
city_col = "city" if "city" in cs.columns else "location"

if city_col == "location":
    cs["city_extracted"] = cs["location"].str.split(",").str[0].str.strip()
    city_col = "city_extracted"

top_cities = (
    cs[cs[city_col].notna()]
      .groupby(city_col)
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
      .head(15)
)

fig = px.bar(
    top_cities[::-1],
    x="count", y=city_col,
    orientation="h",
    title="Top 15 Cities for CS Job Postings",
    labels={city_col: "City", "count": "Postings"},
    color="count",
    color_continuous_scale="Teal"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 7. Company & Demand Stats

In [ ]:
# Top 20 companies by CS job postings
top_companies = (
    cs[cs["name"].notna()]
      .groupby("name")
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
      .head(20)
)

fig = px.bar(
    top_companies[::-1],
    x="count", y="name",
    orientation="h",
    title="Top 20 Companies by CS Job Postings",
    labels={"name": "Company", "count": "Postings"},
    color="count",
    color_continuous_scale="Oranges"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Views-to-applies ratio by subcategory (engagement / conversion)
engagement = (
    cs.dropna(subset=["views", "applies"])
      .assign(ratio=lambda df: df["applies"] / df["views"].clip(lower=1))
      .groupby("category")["ratio"]
      .median()
      .reset_index()
      .rename(columns={"ratio": "apply_rate"})
      .sort_values("apply_rate", ascending=True)
)

fig = px.bar(
    engagement,
    x="apply_rate", y="category",
    orientation="h",
    title="Median Apply-to-View Rate by CS Subcategory",
    labels={"apply_rate": "Applies / Views (median)", "category": "Subcategory"},
    color="apply_rate",
    color_continuous_scale="Greens"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Company size vs. median posting count (bubble chart)
size_order = {
    "1": "1 (1)",
    "2": "2–10",
    "3": "11–50",
    "4": "51–200",
    "5": "201–500",
    "6": "501–1000",
    "7": "1001–5000",
    "8": "5001–10000",
    "9": "10001+",
}

size_dist = (
    cs[cs["company_size"].notna()]
      .groupby("company_size")
      .size()
      .reset_index(name="count")
)
size_dist["size_label"] = size_dist["company_size"].astype(str).map(size_order).fillna(size_dist["company_size"].astype(str))

fig = px.bar(
    size_dist,
    x="size_label", y="count",
    title="CS Job Postings by Company Size",
    labels={"size_label": "Company Size (employees)", "count": "Postings"},
    color="count",
    color_continuous_scale="Oranges"
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [ ]:
# Posting duration (days until expiry)
cs_dur = cs.dropna(subset=["expiry_dt"])
cs_dur = cs_dur.assign(duration_days=(cs_dur["expiry_dt"] - cs_dur["listed_dt"]).dt.days)
cs_dur = cs_dur[(cs_dur["duration_days"] > 0) & (cs_dur["duration_days"] < 365)]

fig = px.histogram(
    cs_dur,
    x="duration_days",
    color="category",
    nbins=60,
    barmode="overlay",
    opacity=0.65,
    title="Posting Duration (Days Until Expiry) by CS Subcategory",
    labels={"duration_days": "Duration (days)", "category": "Category"},
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.show()

---
*End of analysis. All charts are interactive — hover, zoom, and click legend items to filter.*